In [ ]:
pip install protobuf==3.19.6

In [ ]:
pip install tensorflow==2.8.0

In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
print("TensorFlow version:", tf.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
Input_path = '/content/drive/MyDrive/EMG-data.csv'
df = pd.read_csv(Input_path)
print(df.head())
print(df.shape)

In [ ]:
features = df.drop(columns=["label","class","time"])
display(features.head())

In [ ]:
Class = df["class"]
print(Class.unique())

In [ ]:
print(type(Class))
print(type(features))

Class = Class.values
features = features.values

print(type(Class))
print(type(features))

In [44]:
# split in training 70%, validation 10 %,  test 20% test
from sklearn.model_selection import train_test_split
# 80 and 20
x_train, x_test, y_train, y_test = train_test_split(features, Class, test_size=0.2, random_state=1)

In [46]:
# Normalizing data
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)

x_train -= mean
x_train /= std

x_test -= mean
x_test /= std

In [48]:
# one hot encoding Labels
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)

In [57]:
# creating a function for plotting

def plot(loss,val_loss,acc,val_acc):
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(loss) + 1)

    plt.plot(epochs, loss, 'bo', label='Training loss')
    plt.plot(epochs, val_loss, 'b', label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']

    epochs = range(1, len(acc) + 1)

    plt.plot(epochs, acc, 'bo', label='Training acc')
    plt.plot(epochs, val_acc, 'b', label='Validation acc')
    plt.title('Training and validation acc')
    plt.xlabel('Epochs')
    plt.ylabel('acc')
    plt.legend()
    plt.show()

In [58]:
def callbacks(Log,Dir):
  import tensorflow as tf
  import os

  Filepath = Log # Original filepath
  Filepath = Filepath + ".keras" # Adding .keras extension to the filepath # Removing this line, causing the error
  logdir = os.path.join('/content', Dir)

  callbacks_list = [tf.keras.callbacks.TensorBoard(
                    log_dir=logdir,                 #  tensorboard log path
                    histogram_freq=1,),
                    tf.keras.callbacks.EarlyStopping(   # stop if not improving
                    monitor='val_loss',patience=2,),           # monitor validation loss
                    tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss',factor=0.1,         # lr ko .1 se multiply kerdo (kam kerdo)
                    patience=10,),                # reduce the lrate if val loss stop improving
                    tf.keras.callbacks.ModelCheckpoint(
                    filepath= Filepath,             # save model path
                    monitor='val_loss',             # only save best weights
                    save_best_only=True,)]
  return callbacks_list

In [ ]:
from tensorflow.keras import layers, Sequential, optimizers, Input, Model

input_tensor = Input(shape=(8,))
x = layers.Dense(1024, activation='relu')(input_tensor)
y = layers.Dense(512, activation='relu')(x)
z = layers.Dense(256, activation='relu')(y)
z = layers.Dense(128, activation='relu')(z)
z = layers.Dense(64, activation='relu')(z)
z = layers.Dense(32, activation='relu')(z)
z = layers.Dense(128, activation='relu')(y) # acyclic graghs of layers
z = layers.Dense(64, activation='relu')(z)
z = layers.Dense(32, activation='relu')(z)
output_tensor = layers.Dense(8, activation='softmax')(z)

model = Model(input_tensor, output_tensor)

#SGD #RMSprop #Adam #Adadelta #Adagrad ##Adamax ###Nadam #Ftrl
opt = optimizers.Nadam(learning_rate=1e-3)
model.compile(optimizer = opt,
              loss = "categorical_crossentropy",
              metrics = ["accuracy"])

model.summary()

In [ ]:
Path = "model2"
Dir = "my_log_dir"
Call_B_Fun = callbacks(Path,Dir)

batch_size = 512
epochs = 50

history = model.fit(x_train, y_train,
                    batch_size=batch_size, epochs = epochs,
                    validation_split = 0.2, callbacks=Call_B_Fun)

loss = history.history['loss']
val_loss = history.history['val_loss']
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plot(loss,val_loss,acc,val_acc)

In [60]:
model.save('dense_model.h5')
model.save('dense_model.keras')

In [ ]:
from tensorflow.keras.models import load_model
emg = 'dense_model.h5'
emg_model = load_model(emg)
emg_model.summary()

In [ ]:
from tensorflow.keras import layers, Sequential, optimizers, Input, Model

input_tensor = Input(shape=(8,))
x = layers.Dense(1024, activation='relu')(input_tensor)
y = layers.Dense(512, activation='relu')(x)
z = layers.Dense(256, activation='relu')(y)
z = layers.Dense(128, activation='relu')(z)
z = layers.Dense(64, activation='relu')(z)
z = layers.Dense(32, activation='relu')(z)
z = layers.Dense(128, activation='relu')(y) # acyclic graghs of layers
z = layers.Dense(64, activation='relu')(z)
z = layers.Dense(32, activation='relu')(z)
output_tensor = layers.Dense(8, activation='softmax')(z)

model = Model(input_tensor, output_tensor)

opt = optimizers.Nadam(learning_rate=1e-3)
model.compile(optimizer = opt,
              loss = "categorical_crossentropy",
              metrics = ["accuracy"])

model.set_weights(emg_model.get_weights())   # using pretrained model weights

model.summary()

In [ ]:
callbacks_list = [tf.keras.callbacks.EarlyStopping(   # stop if not improving
                monitor='acc',patience=5,mode='max'),           # monitor validation accuracy
                tf.keras.callbacks.ModelCheckpoint(
                filepath='my_model.keras',
                monitor='val_loss',                   # only save best weights
                save_best_only=True,)]                # when vall loss is improved

batch_size = 512
epochs = 20

history = model.fit(x_train, y_train,
                    batch_size=batch_size, epochs = epochs,
                    validation_split = 0.2,
                    callbacks=callbacks_list)

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plot(loss,val_loss,acc,val_acc)

In [ ]:
model.save('dense_model_1.h5')
model.save('dense_model_1.keras')

In [63]:
# Define training function
@tf.function(input_signature=[tf.TensorSpec([None, 8], tf.float32), tf.TensorSpec([None, 8], tf.float32)])
def train_step(x, y):
    with tf.GradientTape() as tape:
        prediction = model(x, training=True)
        loss = tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y, prediction))
    gradients = tape.gradient(loss, model.trainable_variables)
    model.optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return {"loss": loss}

@tf.function(input_signature=[
    tf.TensorSpec(shape=[None, 8], dtype=tf.float32, name="input")
])
def serve(input):
    input = (input - mean) / std
    return {"output": model(input)}

# Save the model with training and inference signatures
SAVED_MODEL_DIR = "my_saved_model"
tf.saved_model.save(
    model,
    SAVED_MODEL_DIR,
    signatures={
        "train": train_step,
        "serving_default": serve
    }
)

# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_saved_model("my_saved_model")
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter.experimental_enable_resource_variables = True
tflite_model = converter.convert()

# Save TFLite model
with open('dense_model_with_train_signature.tflite', 'wb') as f:
    f.write(tflite_model)


In [64]:
import tensorflow as tf
model = tf.keras.models.load_model('/content/dense_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter.experimental_enable_resource_variables = True

# Add the signatures to the converter
# converter.signatures = signatures

# Convert the model
tflite_model = converter.convert()

# Save the model to a file
with open('dense_model.tflite', 'wb') as f:
    f.write(tflite_model)

In [65]:
import tensorflow as tf

# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path="/content/dense_model_with_train_signature.tflite")

# Get the signature list
signatures = interpreter.get_signature_list()

# Print the signature list
print("Signature List:", signatures)

Signature List: {'serving_default': {'inputs': ['input'], 'outputs': ['output']}, 'train': {'inputs': ['x', 'y'], 'outputs': ['loss']}}
